In [1]:
import mne
from scipy.signal import butter, lfilter, iirnotch
from scipy.signal import welch
import numpy as np
import fooof
import matplotlib.pyplot as plt
from datetime import datetime
import pandas as pd
import warnings

C:\Users\86159\AppData\Local\Temp\ipykernel_13588\3920452700.py:5: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  import fooof


In [2]:
# 读取EDF格式的脑电信号数据
file_path = 'cairangzhaxi\SEEG\cairangzhaxi_SZ1.edf'
csv_path='F:\datasets\发严子博兰大二院数据及标签1\标签\间期数据\cairangzhaxi\sz1.xlsx'
raw = mne.io.read_raw_edf(file_path,encoding='latin1',preload=True)
channel_name=raw.info['ch_names']
s_freqs=raw.info['sfreq']
num_chs=len(channel_name)
print(num_chs)

Extracting EDF parameters from e:\datasets\发严子博兰大二院数据及标签1\标签\数据SEEG+EEG\cairangzhaxi\SEEG\cairangzhaxi_SZ1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 984193  =      0.000 ...   492.096 secs...


C:\Users\86159\AppData\Local\Temp\ipykernel_13588\2362823229.py:4: RuntimeWarning: Channel names are not unique, found duplicates for: {'POL A5A6'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path,encoding='latin1',preload=True)


93


In [3]:
# 1000Hz低通滤波 
raw.filter(None, 999, fir_design='firwin') 
# 去除50Hz及其倍数的工频 
raw.notch_filter(np.arange(50, s_freqs // 2 - 1, 50), fir_design='firwin',notch_widths=4)

Filtering raw data in 1 contiguous segment
Setting up low-pass filter at 1e+03 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal lowpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Upper passband edge: 999.00 Hz
- Upper transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 999.50 Hz)
- Filter length: 6601 samples (3.300 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    1.5s


Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 13201 samples (6.601 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.4s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    2.0s


Measurement date,"March 06, 2019 19:09:36 GMT"
Experimenter,Unknown
Participant,X
Digitized points,Not available
Good channels,93 EEG
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,2000.00 Hz
Highpass,0.00 Hz
Lowpass,999.00 Hz


In [4]:
#计算癫痫发作期的相对时间
sz_st ="9:06:04"
sz_ed ="9:06:20"
start_h=raw.info['meas_date'].hour
start_m=raw.info['meas_date'].minute
start_s=raw.info['meas_date'].second
start=f"{start_h}:{start_m}:{start_s}"
time1 = datetime.strptime(start, "%H:%M:%S")
time2 = datetime.strptime(sz_st, "%H:%M:%S")
time3 = datetime.strptime(sz_ed, "%H:%M:%S")
sz_S =(time2-time1).total_seconds()
sz_T =(time3-time1).total_seconds()

print(sz_S,sz_T)

-36212.0 -36196.0


In [25]:
start_sample = int(sz_S * s_freqs)
end_sample = int(sz_T * s_freqs)
eeg_data = raw.get_data()[:, start_sample:end_sample]

In [26]:
#计算功率谱密度并处理工频处
freqs, psd = welch(eeg_data, fs=s_freqs, nperseg=s_freqs)
cols_to_replace = np.arange(50, psd.shape[1]-50, 50)
for col in cols_to_replace:
    for i in range(-3,3):
        t_col=col+i
        left_col = psd[:, t_col - 1]
        right_col = psd[:, t_col + 1]
        mean_col = (left_col + right_col) / 2
        psd[:, t_col] = mean_col

In [27]:
delta=[0.5,4]
theta=[4,8]
alpha=[8,13]
beta=[13,30]
l_gamma=[30,80]
hf=[80,500]
def extract_band_power(freqs,psd,band):
    band_mask = (freqs >= band[0]) & (freqs <= band[1])
    band_power = np.mean(psd[:, band_mask], axis=1)
    return band_power

In [28]:
delta_p=extract_band_power(freqs,psd,delta)
theta_p=extract_band_power(freqs,psd,theta)
alpha_p=extract_band_power(freqs,psd,alpha)
beta_p=extract_band_power(freqs,psd,beta)
l_gamma_p=extract_band_power(freqs,psd,l_gamma)
hf_p=extract_band_power(freqs,psd,hf)

In [29]:
warnings.filterwarnings("ignore", category=UserWarning, module='fooof')
fitlist=[]
fm = fooof.FOOOF(peak_threshold=1,min_peak_height=0.25,max_n_peaks=30)
for i in range(0,num_chs):
    fm.fit(freqs,psd[i],[0,500])
    slope=-fm.aperiodic_params_[1]
    error = fm.get_params('error')
    r_squared = fm.get_params('r_squared')
    print(f"拟合错误率: {error}",f"R方值: {r_squared}")
    fitlist.append(slope)


FOOOF WARNING: Skipping frequency == 0, as this causes a problem with fitting.

FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.50 <= 1.00
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2x the frequency resolution.

拟合错误率: 0.10924733456328044 R方值: 0.9612971113637022

FOOOF WARNING: Skipping frequency == 0, as this causes a problem with fitting.

FOOOF WARNING: Lower-bound peak width limit is < or ~= the frequency resolution: 0.50 <= 1.00
	Lower bounds below frequency-resolution have no effect (effective lower bound is the frequency resolution).
	Too low a limit may lead to overfitting noise as small bandwidth peaks.
	We recommend a lower bound of approximately 2x the frequency resolution.

拟合错误率: 0.11382118828606613 R方值: 0.9567723867048028

FOOOF WARNING: Skipping frequency

In [30]:
df=pd.DataFrame({
    'channel_name':channel_name,
    'delta':delta_p,
    'theta':theta_p,
    'alpha':alpha_p,
    'beta':beta_p,
    'l_gamma':l_gamma_p,
    'hf':hf_p,
    'slope':fitlist
})
del eeg_data
df.to_excel(csv_path,index=False)